# Optimizers — Exercises

Companion to the note [Optimizers](Optimizers.md).

Practise the optimizers from the note's table (SGD + momentum, Nesterov, AdaGrad, RMSProp, Adam, AdamW) and the practical
advice (learning rates, warm-up + cosine decay): first by hand (effective step sizes, bias correction, stability limits), then by implementing
each update rule from scratch in NumPy and racing them on an ill-conditioned quadratic and on logistic regression. Builds on [[Gradient Descent]].

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import expit
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)

# Ill-conditioned quadratic f(w) = 1/2 (w1^2 + 100 w2^2), condition number 100
A_q = np.diag([1.0, 100.0])
f_q = lambda w: 0.5 * w @ A_q @ w
grad_q = lambda w: A_q @ w
w0_q = np.array([1.0, 1.0])

def run(opt, grad, w0, steps):
    # generic driver: opt.step(w, g) returns the new w
    w = np.array(w0, float)
    for _ in range(steps):
        w = opt.step(w, grad(w))
    return w

## Part A · Concepts

### Exercise 1 · Momentum in a ravine, and Nesterov's look-ahead
*🧠 Concept · ★☆☆*

1. Plain GD on a long narrow valley (ill-conditioned loss) zig-zags across the valley and crawls along it. Explain how a velocity
   $v \leftarrow \beta v + g$, $w\leftarrow w - \eta v$ fixes both problems.
2. What does Nesterov momentum do differently, and why does it help?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Across the valley the gradient flips sign every step; along the valley it points the same way every step. What happens to their running sums?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Velocity is an exponentially weighted sum of past gradients. Components that **flip sign** (across the valley) cancel, damping the zig-zag;
   components that **agree** (along the valley) accumulate up to $1/(1-\beta)$ times their size (Exercise 4), so progress along the valley speeds up.
2. Nesterov evaluates the gradient at the **look-ahead** point $w - \eta\beta v$ (where momentum is about to carry us) instead of at $w$.
   If momentum is overshooting, the look-ahead gradient already points back and corrects earlier: less oscillation, and provably faster rates for convex problems.

</details>

### Exercise 2 · Adam + L2 vs. AdamW
*🧠 Concept · ★★☆*

For SGD, adding $\frac\lambda2\lVert w\rVert^2$ to the loss and "weight decay" $w\leftarrow(1-\eta\lambda)w$ are the same thing. For Adam they are not.
Explain why, and why AdamW (decoupled weight decay) is the default for [[Transformers]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

With L2 in the loss, the decay term $\lambda w$ enters $g$ and is then divided by $\sqrt{\hat v}$. Which weights get decayed most?

</details>

<details>
<summary><b>✅ Solution</b></summary>

With Adam + L2, the penalty gradient $\lambda w$ is mixed into $g$ and then rescaled per parameter by $1/\sqrt{\hat v}$. Weights with large historic gradients
(large $\hat v$) are decayed **less**; weights with tiny gradients are decayed **more** (the normalisation turns $\lambda w$ into a step of size $\approx\eta$).
So the regularisation strength depends on gradient statistics instead of being uniform. AdamW applies $w \leftarrow w - \eta\lambda w$ **outside** the adaptive
update, so every weight shrinks by the same factor, as intended. Empirically it generalises better and makes $\lambda$ and $\eta$ easier to tune (Exercise 13 demonstrates it).

</details>

### Exercise 3 · Practical recipe
*🧠 Concept · ★★☆*

1. Why do we use learning-rate **warm-up**, especially with Adam and large batches?
2. Why decay the learning rate at the end of training (cosine/linear)?
3. The note says SGD + momentum "often generalizes better for vision". Give one possible reason, and one reason people still default to Adam.
4. When would you use gradient clipping (norm 1.0)?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Early in training, Adam's $\hat v$ is estimated from very few gradients; the network is also far from any reasonable region.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. At the start the weights are random, gradients are large and erratic, and Adam's second-moment estimate $\hat v$ is based on few samples (noisy, often too small → huge steps).
   A short linear ramp from ~0 to $\eta_{\max}$ prevents early divergence and lets normalisation statistics settle.
2. With a constant rate SGD bounces around the minimum at a noise level $\propto\eta$; decaying $\eta$ lets it settle into the basin (and lowers the final loss noticeably).
3. SGD's larger, isotropic noise is thought to bias it toward flatter minima, which tend to generalise; Adam's per-parameter scaling can find sharper ones.
   Adam is popular because it works with little tuning across architectures (default $\eta = 10^{-3}$) and handles sparse/badly scaled gradients.
4. For models with occasional gradient spikes: [[RNN and LSTM]] (exploding gradients through time) and [[Transformers]] (loss spikes); clipping caps the step without changing its direction ([[Training Tricks]]).

</details>

## Part B · By hand

### Exercise 4 · Effective learning rate of momentum
*✍️ By hand · ★☆☆*

With $v_t = \beta v_{t-1} + g$ and a **constant** gradient $g$, show that $v_t \to g/(1-\beta)$. By what factor is the step size amplified for
$\beta = 0.9$ and $\beta=0.99$? How many steps does it take until $v_t$ reaches 90% of its limit for $\beta=0.9$?

In [ ]:
amp_09 = None
amp_099 = None
steps_90pct = None

_v, _n = 0.0, 0
while _v < 0.9 * 10:
    _v = 0.9 * _v + 1; _n += 1
check('amplification beta=0.9', amp_09, sum(0.9 ** k for k in range(10000)))
check('amplification beta=0.99', amp_099, sum(0.99 ** k for k in range(100000)))
check('steps to 90%', steps_90pct, _n)

<details>
<summary><b>💡 Hint</b></summary>

$v_t = g(1 + \beta + \dots + \beta^{t-1}) = g\frac{1-\beta^t}{1-\beta}$. Solve $1-\beta^t\ge0.9$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$v_t = g\,\frac{1-\beta^t}{1-\beta}\to\frac{g}{1-\beta}$: amplification $10\times$ for $\beta=0.9$, $100\times$ for $\beta=0.99$.
90% needs $0.9^t\le0.1$, i.e. $t\ge\log0.1/\log0.9 = 21.85$, so **22 steps**. Hence when switching on momentum you usually lower $\eta$ accordingly.

```python
amp_09 = 10
amp_099 = 100
steps_90pct = 22
```

</details>

### Exercise 5 · Why Adam needs bias correction
*✍️ By hand · ★★☆*

Adam: $m_t = \beta_1m_{t-1} + (1-\beta_1)g_t$, $v_t = \beta_2v_{t-1} + (1-\beta_2)g_t^2$, $\hat m_t = m_t/(1-\beta_1^t)$, $\hat v_t = v_t/(1-\beta_2^t)$,
$w\leftarrow w - \eta\,\hat m_t/(\sqrt{\hat v_t}+\epsilon)$, with $m_0=v_0=0$ and the note's defaults.

For the very first step with gradient $g_1 = 0.02$ and $\eta = 10^{-3}$, compute the step size (a) with bias correction and (b) **without** it ($m_1/(\sqrt{v_1}+\epsilon)$).

In [ ]:
step_corrected = None
step_uncorrected = None

_g, _eta, _eps = 0.02, 1e-3, 1e-8
_m, _v = (1 - 0.9) * _g, (1 - 0.999) * _g ** 2
check('with correction', step_corrected, _eta * (_m / 0.1) / (np.sqrt(_v / 0.001) + _eps), tol=1e-7)
check('without correction', step_uncorrected, _eta * _m / (np.sqrt(_v) + _eps), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

$m_1 = 0.1g$, $v_1 = 0.001g^2$. With correction both become exactly $g$ and $g^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\hat m_1 = g$, $\hat v_1 = g^2$, step $= \eta\,g/(|g|+\epsilon) \approx \eta = 10^{-3}$ (exactly $9.999995\times10^{-4}$). The first Adam step is $\approx\eta\,\mathrm{sign}(g)$, whatever the gradient's scale.
(b) $m_1 = 0.002$, $\sqrt{v_1} = 6.32\times10^{-4}$: step $=10^{-3}\cdot0.002/6.32\times10^{-4} = 3.16\times10^{-3}$, i.e. $\frac{1-\beta_1}{\sqrt{1-\beta_2}} = 3.16\times$ too large.
Without correction both moments are biased toward their zero init; since $\beta_2$ is closer to 1, $v$ is biased more and the early steps are too big.

```python
step_corrected = 1e-3
step_uncorrected = 3.162e-3
```

</details>

### Exercise 6 · AdaGrad by hand
*✍️ By hand · ★☆☆*

AdaGrad: $G_t = G_{t-1} + g_t^2$, $w \leftarrow w - \eta\,g_t/\sqrt{G_t}$ (ignore $\epsilon$). Start at $w_0 = 0$ with $\eta = 1$ and constant gradient $g_t = 1$.
Compute $w_3$. What happens to the step size as $t\to\infty$, and why is that a problem for deep networks?

In [ ]:
w3_adagrad = None

check('w_3', w3_adagrad, -sum(1 / np.sqrt(t) for t in (1, 2, 3)), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$G_t = t$, so the $t$-th step is $1/\sqrt t$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$w_3 = -(1 + 1/\sqrt2 + 1/\sqrt3) = -(1 + 0.7071 + 0.5774) = -2.2845$.
The step $\eta/\sqrt t\to0$ because $G_t$ only grows: AdaGrad's learning rate decays forever. Good for convex problems with sparse features,
bad for long non-convex training where it stalls. RMSProp replaces the sum with an exponential moving average so the rate can stay constant.

```python
w3_adagrad = -2.2845
```

</details>

### Exercise 7 · Stability limit on an ill-conditioned quadratic
*✍️ By hand · ★★☆*

For $f(w) = \tfrac12(w_1^2 + 100w_2^2)$ (the `f_q` in the setup), plain GD gives $w_i \leftarrow (1-\eta\lambda_i)w_i$ with $\lambda = (1, 100)$.

1. What is the largest learning rate for which GD converges? What is the condition number $\kappa$?
2. With $\eta = 0.019$, after how many steps has $w_1$ shrunk below $10^{-3}$ of its start?

In [ ]:
eta_max = None
kappa = None
steps_w1 = None

_lam = np.linalg.eigvalsh(A_q)
check('eta_max', eta_max, 2 / _lam.max())
check('kappa', kappa, _lam.max() / _lam.min())
_w, _n = 1.0, 0
while abs(_w) >= 1e-3:
    _w *= (1 - 0.019 * 1); _n += 1
check('steps', steps_w1, _n)

<details>
<summary><b>💡 Hint</b></summary>

Need $|1-\eta\lambda_i| < 1$ for all $i$. For (2) solve $0.981^t < 10^{-3}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $|1-100\eta|<1 \Rightarrow \eta < 2/100 = 0.02$. $\kappa = 100/1 = 100$.
2. $t > \log(10^{-3})/\log(0.981) = 360.1$, so **361** steps, although $w_2$ converged after a few. The stiff direction caps $\eta$, the flat one then needs $\sim\kappa$ steps.
   Momentum improves this to $\sim\sqrt\kappa$; adaptive methods rescale each coordinate (Exercises 10–11).

```python
eta_max = 0.02
kappa = 100
steps_w1 = 361
```

</details>

### Exercise 8 · The bias of an exponential moving average
*✍️ By hand · ★☆☆*

An EMA $v_t = \beta v_{t-1} + (1-\beta)x$ starts at $v_0 = 0$ and sees a constant $x$. Show $v_t = (1-\beta^t)x$.
For $\beta = 0.999$ (Adam's $\beta_2$), what fraction of $x$ has $v_{10}$ reached? And after how many steps is it 63% ($1 - e^{-1}$)?

In [ ]:
frac_10 = None
steps_63 = None

_v = 0.0
for _ in range(10):
    _v = 0.999 * _v + 0.001
check('fraction after 10 steps', frac_10, _v, tol=1e-6)
check('steps to 63%', steps_63, int(np.ceil(np.log(np.exp(-1)) / np.log(0.999))), tol=1)

<details>
<summary><b>💡 Hint</b></summary>

Unroll: $v_t = (1-\beta)x\sum_{k<t}\beta^k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$v_t = (1-\beta)x\frac{1-\beta^t}{1-\beta} = (1-\beta^t)x$. $v_{10} = (1 - 0.999^{10})x = 0.00996\,x$: under 1%!
$1-\beta^t = 1-e^{-1}$ when $t = -1/\ln\beta \approx 1000$ steps (the EMA's "time constant" $\approx 1/(1-\beta)$). Dividing by $1-\beta^t$ removes exactly this bias.

```python
frac_10 = 1 - 0.999 ** 10
steps_63 = 1000
```

</details>

### Exercise 9 · Warm-up + cosine schedule by hand
*✍️ By hand · ★☆☆*

A schedule warms up linearly from 0 to $\eta_{\max} = 10^{-3}$ during the first 100 steps, then decays with a cosine to 0 at step 1000:
$\eta_t = \eta_{\max}\cdot\tfrac12\big(1 + \cos(\pi\,\tfrac{t-100}{900})\big)$ for $t\ge100$.
Compute $\eta_{50}$, $\eta_{325}$ and $\eta_{550}$.

In [ ]:
lr_50 = None
lr_325 = None
lr_550 = None

_cos = lambda t: 1e-3 * 0.5 * (1 + np.cos(np.pi * (t - 100) / 900))
check('lr at 50', lr_50, 1e-3 * 50 / 100, tol=1e-7)
check('lr at 325', lr_325, _cos(325), tol=1e-7)
check('lr at 550', lr_550, _cos(550), tol=1e-7)

<details>
<summary><b>💡 Hint</b></summary>

Progress through the decay phase at $t=325$ is $225/900 = 1/4$; $\cos(\pi/4) = 0.7071$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\eta_{50} = 0.5\,\eta_{\max} = 5\times10^{-4}$. $\eta_{325} = 10^{-3}\cdot\tfrac12(1+0.7071) = 8.536\times10^{-4}$.
$\eta_{550}$: progress $1/2$, $\cos(\pi/2) = 0$, so $5\times10^{-4}$. The cosine stays high for a long time, then decays smoothly to 0.

```python
lr_50 = 5e-4
lr_325 = 8.536e-4
lr_550 = 5e-4
```

</details>

## Part C · Code from scratch

### Exercise 10 · SGD, momentum and Nesterov
*💻 Code · ★☆☆*

Implement three optimizer classes with a method `step(w, g)` that returns the updated weights (keep any state on `self`):

- `SGD(lr)`: $w - \eta g$
- `Momentum(lr, beta)`: $v\leftarrow\beta v + g$, $w\leftarrow w-\eta v$
- `Nesterov(lr, beta)` (PyTorch form): $v\leftarrow\beta v + g$, $w\leftarrow w-\eta(g + \beta v)$

Run each for 100 steps on the quadratic `f_q` from `w0_q` using the `run` driver: SGD with its best rate $\eta=0.019$ (Exercise 7), momentum and Nesterov with
$\eta=0.01$, $\beta = 0.8$. Store the final losses in `final_loss` (dict). Then try Nesterov with $\eta = 0.019$: what happens?

In [ ]:
class SGD:
    def __init__(self, lr): self.lr = lr
    def step(self, w, g): return None

class Momentum:
    def __init__(self, lr, beta=0.9): self.lr, self.beta, self.v = lr, beta, None
    def step(self, w, g): return None

class Nesterov:
    def __init__(self, lr, beta=0.9): self.lr, self.beta, self.v = lr, beta, None
    def step(self, w, g): return None

final_loss = None   # {'sgd': ..., 'momentum': ..., 'nesterov': ...}

_g = np.array([0.3, -2.0]); _w = np.array([1.0, 1.0])
check('SGD step', SGD(0.1).step(_w, _g), _w - 0.1 * _g)
check('first momentum step equals SGD', Momentum(0.1).step(_w, _g), _w - 0.1 * _g)
if final_loss is None:
    check('momentum beats SGD by > 100x after 100 steps', None, True)
else:
    print(final_loss)
    check('momentum beats SGD by > 100x after 100 steps', final_loss['momentum'] < final_loss['sgd'] / 100 and final_loss['nesterov'] < final_loss['sgd'] / 100, True)

<details>
<summary><b>💡 Hint</b></summary>

Initialise `self.v = np.zeros_like(w)` on the first call. Return a new array (`w - ...`) rather than modifying `w` in place.

</details>

<details>
<summary><b>✅ Solution</b></summary>

SGD after 100 steps still has $w_1 = 0.981^{100}\approx0.147$ (loss $\approx 0.011$); momentum ($\approx5\times10^{-7}$) and Nesterov ($\approx1.5\times10^{-6}$) are
four orders of magnitude lower, because the velocity builds up along the flat direction (Exercise 4).
With $\eta=0.019$ Nesterov **diverges**: its update $\eta(g + \beta v)$ effectively applies the stiff-direction gradient with a larger weight, so its stability
limit on $\eta$ is lower than heavy-ball momentum's. Momentum methods need the learning rate re-tuned (typically lowered) compared with plain SGD.

```python
class SGD:
    def __init__(self, lr): self.lr = lr
    def step(self, w, g): return w - self.lr * g

class Momentum:
    def __init__(self, lr, beta=0.9): self.lr, self.beta, self.v = lr, beta, None
    def step(self, w, g):
        self.v = g.copy() if self.v is None else self.beta * self.v + g
        return w - self.lr * self.v

class Nesterov:
    def __init__(self, lr, beta=0.9): self.lr, self.beta, self.v = lr, beta, None
    def step(self, w, g):
        self.v = g.copy() if self.v is None else self.beta * self.v + g
        return w - self.lr * (g + self.beta * self.v)

final_loss = {name: f_q(run(opt, grad_q, w0_q, 100)) for name, opt in
              [('sgd', SGD(0.019)), ('momentum', Momentum(0.01, 0.8)), ('nesterov', Nesterov(0.01, 0.8))]}
print('Nesterov with lr=0.019:', f_q(run(Nesterov(0.019), grad_q, w0_q, 100)))
print(final_loss)
```

</details>

### Exercise 11 · Adam from scratch
*💻 Code · ★★☆*

Implement `Adam(lr=1e-3, b1=0.9, b2=0.999, eps=1e-8)` with bias correction (same `step(w, g)` interface).
Check the property from Exercise 5 (the first step is $\approx\eta\,\mathrm{sign}(g)$ for every coordinate, regardless of scale), then run it on `f_q`
with `lr=0.1` for 300 steps and store the final loss in `adam_loss`.

In [ ]:
class Adam:
    def __init__(self, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps
        self.m = self.v = None
        self.t = 0
    def step(self, w, g):
        return None

adam_loss = None

_g = np.array([1e-4, -3.0, 250.0]); _w = np.zeros(3)
_s = Adam(lr=0.01).step(_w, _g)
check('first step = -lr * sign(g)', None if _s is None else _s - _w, -0.01 * np.sign(_g), tol=1e-5)
check('Adam converges on f_q', None if adam_loss is None else adam_loss < 1e-4, True)

<details>
<summary><b>💡 Hint</b></summary>

Increment `self.t` first; `m_hat = self.m / (1 - self.b1 ** self.t)`, same for `v`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The scale invariance (a gradient of $10^{-4}$ and one of $250$ both give a step of $\eta$) is Adam's key property: $\eta$ becomes a step size in
parameter units, which makes the default $10^{-3}$ work across very different models. On `f_q` it converges despite $\kappa = 100$.

```python
class Adam:
    def __init__(self, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps
        self.m = self.v = None
        self.t = 0
    def step(self, w, g):
        if self.m is None:
            self.m, self.v = np.zeros_like(w, dtype=float), np.zeros_like(w, dtype=float)
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g ** 2
        m_hat = self.m / (1 - self.b1 ** self.t)
        v_hat = self.v / (1 - self.b2 ** self.t)
        return w - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

adam_loss = f_q(run(Adam(lr=0.1), grad_q, w0_q, 300))
print(adam_loss)
```

</details>

### Exercise 12 · AdaGrad vs. RMSProp under a constant gradient
*💻 Code · ★★☆*

Implement `AdaGrad(lr, eps=1e-8)` ($G \mathrel{+}= g^2$) and `RMSProp(lr, rho=0.9, eps=1e-8)` ($s\leftarrow\rho s + (1-\rho)g^2$), both with
$w\leftarrow w - \eta\,g/(\sqrt{\cdot}+\epsilon)$. Feed each a **constant** gradient $g = 1$ for 50 steps from $w=0$ with $\eta=0.1$ and record the step
sizes $|\Delta w_t|$ in `steps_adagrad` and `steps_rmsprop`. Derive the closed forms and compare.

In [ ]:
class AdaGrad:
    def __init__(self, lr, eps=1e-8): self.lr, self.eps, self.G = lr, eps, None
    def step(self, w, g): return None

class RMSProp:
    def __init__(self, lr, rho=0.9, eps=1e-8): self.lr, self.rho, self.eps, self.s = lr, rho, eps, None
    def step(self, w, g): return None

steps_adagrad = None
steps_rmsprop = None

_t = np.arange(1, 51)
check('AdaGrad steps = lr / sqrt(t)', steps_adagrad, 0.1 / np.sqrt(_t), tol=1e-6)
check('RMSProp steps = lr / sqrt(1 - rho^t)', steps_rmsprop, 0.1 / np.sqrt(1 - 0.9 ** _t), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Collect `abs(w_new - w)` in a loop of 50 calls with `g = np.array([1.0])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

AdaGrad: $G_t = t$, step $\eta/\sqrt t$, decaying forever ($0.1\to0.014$ after 50 steps).
RMSProp: $s_t = 1-\rho^t$, step $\eta/\sqrt{1-\rho^t}$, which starts **large** ($0.1/\sqrt{0.1} = 0.316$, the uncorrected-EMA effect from Exercise 5) and then settles at $\eta$.
Adam = RMSProp + momentum + the bias correction that removes that initial overshoot.

```python
class AdaGrad:
    def __init__(self, lr, eps=1e-8): self.lr, self.eps, self.G = lr, eps, None
    def step(self, w, g):
        self.G = g ** 2 if self.G is None else self.G + g ** 2
        return w - self.lr * g / (np.sqrt(self.G) + self.eps)

class RMSProp:
    def __init__(self, lr, rho=0.9, eps=1e-8): self.lr, self.rho, self.eps, self.s = lr, rho, eps, None
    def step(self, w, g):
        s_prev = np.zeros_like(g) if self.s is None else self.s
        self.s = self.rho * s_prev + (1 - self.rho) * g ** 2
        return w - self.lr * g / (np.sqrt(self.s) + self.eps)

def _record(opt, n=50):
    w, out = np.zeros(1), []
    for _ in range(n):
        w_new = opt.step(w, np.array([1.0]))
        out.append(abs(w_new - w)[0]); w = w_new
    return np.array(out)

steps_adagrad = _record(AdaGrad(0.1))
steps_rmsprop = _record(RMSProp(0.1))
```

</details>

### Exercise 13 · AdamW vs. Adam + L2
*💻 Code · ★★★*

Demonstrate Exercise 2 numerically. Take a single weight $w_0 = 1$ whose **data-loss gradient is zero** (it is irrelevant to the task), $\eta = 0.01$, $\lambda = 0.1$, 100 steps:

- **Adam + L2**: feed Adam the gradient $g = \lambda w$.
- **AdamW**: $w\leftarrow w - \eta\lambda w$, then the Adam step with data gradient $g = 0$.

Store the final weights in `w_adam_l2` and `w_adamw`. Which one behaves like "weight decay"? Use your `Adam` class.

In [ ]:
w_adam_l2 = None
w_adamw = None

check('AdamW = exact exponential decay', w_adamw, (1 - 0.01 * 0.1) ** 100)
check('Adam+L2 shrinks ~lr per step regardless of lambda', None if w_adam_l2 is None else abs(w_adam_l2) < 0.3, True)

<details>
<summary><b>💡 Hint</b></summary>

For AdamW with $g=0$, Adam's $m$ stays 0, so only the decoupled decay acts: $w_{100} = (1-\eta\lambda)^{100}$. For Adam + L2, remember the first steps are $\approx\eta\,\mathrm{sign}(g)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

AdamW: $w_{100} = 0.999^{100} = 0.905$, a gentle exponential decay controlled by $\eta\lambda$ exactly as intended.
Adam + L2: Adam normalises $g = \lambda w$, so each step is $\approx\eta$ **whatever $\lambda$ is** (try $\lambda = 10^{-4}$: almost the same result!); the weight drops roughly linearly
toward 0 (here to $\approx 0.22$ in 100 steps, versus 0.905 for AdamW). The penalty no longer acts like a proportional shrinkage: for weights with small data gradients it is far too strong,
for weights with large gradients (large $\hat v$) far too weak. Decoupling fixes that.

```python
opt = Adam(lr=0.01); w = np.array([1.0])
for _ in range(100):
    w = opt.step(w, 0.1 * w)
w_adam_l2 = w[0]

opt = Adam(lr=0.01); w = np.array([1.0])
for _ in range(100):
    w = w - 0.01 * 0.1 * w
    w = opt.step(w, np.zeros(1))
w_adamw = w[0]
print(w_adam_l2, w_adamw)
```

</details>

### Exercise 14 · Optimizer race on logistic regression
*💻 Code · ★★☆*

Minimise the L2-regularised logistic loss on the standardised breast-cancer data,
$J(w,b) = \frac1N\sum_n \mathrm{BCE}(\sigma(w^\top x_n+b), y_n) + \frac{1}{2CN}\lVert w\rVert^2$ with $C=1$
(this is exactly sklearn's `LogisticRegression(C=1)` objective divided by $CN$; the bias is not penalised).
Run 500 full-batch steps with `SGD(0.5)`, `Momentum(0.5, 0.9)`, `Adam(0.05)` and store the final objective values in `race`.
Compare with $J$ at sklearn's solution, `J_ref`.

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True)
Xc = StandardScaler().fit_transform(Xc)
N_c, C_reg = len(yc), 1.0

def J_and_grad(theta):
    # theta = [b, w_1, ..., w_d]; return (J, gradient)
    return None, None

_sk = LogisticRegression(C=C_reg, max_iter=10000, tol=1e-10).fit(Xc, yc)
theta_ref = np.r_[_sk.intercept_, _sk.coef_.ravel()]
J_ref = None if J_and_grad(theta_ref)[0] is None else J_and_grad(theta_ref)[0]
race = None   # {'sgd': ..., 'momentum': ..., 'adam': ...}

if race is None:
    check('best optimizer within 1e-3 of the sklearn optimum', None, True)
else:
    print({k: round(v, 5) for k, v in race.items()}, 'J_ref =', round(J_ref, 5))
    check('best optimizer within 1e-3 of the sklearn optimum', min(race.values()) - J_ref < 1e-3 and min(race.values()) > J_ref - 1e-6, True)

<details>
<summary><b>💡 Hint</b></summary>

With $p = \sigma(Xw+b)$: $\nabla_w J = X^\top(p-y)/N + w/(CN)$, $\partial J/\partial b = \overline{p - y}$. Use `np.logaddexp(0, z) - y*z` for a stable BCE.

</details>

<details>
<summary><b>✅ Solution</b></summary>

All three get close to sklearn's optimum (L-BFGS), with momentum and Adam reaching it within $\sim10^{-4}$ while plain GD is still slowly creeping down the flat directions
(the standardised but correlated features make the problem ill-conditioned). None can go *below* $J_{\text{ref}}$, a nice sanity check of your gradient and of sklearn's objective.

```python
Xc, yc = load_breast_cancer(return_X_y=True)
Xc = StandardScaler().fit_transform(Xc)
N_c, C_reg = len(yc), 1.0

def J_and_grad(theta):
    b, w = theta[0], theta[1:]
    z = Xc @ w + b
    J = np.mean(np.logaddexp(0, z) - yc * z) + w @ w / (2 * C_reg * N_c)
    r = expit(z) - yc
    return J, np.r_[r.mean(), Xc.T @ r / N_c + w / (C_reg * N_c)]

_sk = LogisticRegression(C=C_reg, max_iter=10000, tol=1e-10).fit(Xc, yc)
theta_ref = np.r_[_sk.intercept_, _sk.coef_.ravel()]
J_ref = J_and_grad(theta_ref)[0]
race = {}
for name, opt in [('sgd', SGD(0.5)), ('momentum', Momentum(0.5, 0.9)), ('adam', Adam(0.05))]:
    th = run(opt, lambda t: J_and_grad(t)[1], np.zeros(Xc.shape[1] + 1), 500)
    race[name] = float(J_and_grad(th)[0])
print(race, J_ref)
```

</details>

### Exercise 15 · A warm-up + cosine scheduler
*💻 Code · ★☆☆*

Implement `lr_at(t, lr_max, warmup, total, lr_min=0.0)`: linear warm-up from 0 to `lr_max` over `warmup` steps, then cosine decay to `lr_min` at `total`.
Plot (optional) the schedule for `lr_max=1e-3, warmup=100, total=1000` and confirm your by-hand values from Exercise 9.

In [ ]:
def lr_at(t, lr_max, warmup, total, lr_min=0.0):
    return None

sched = [lr_at(t, 1e-3, 100, 1000) for t in range(1001)]

if sched[0] is None:
    check('schedule', None, 0)
else:
    _s = np.array(sched, float)
    check('peak at end of warm-up', _s.argmax(), 100)
    check('starts at 0 and ends at lr_min', [_s[0], _s[-1], lr_at(1000, 1e-3, 100, 1000, lr_min=1e-5)], [0, 0, 1e-5], tol=1e-9)
    check('monotone up, then monotone down', bool(np.all(np.diff(_s[:101]) > 0) and np.all(np.diff(_s[100:]) < 0)), True)
    check('midpoint of decay = (max + min) / 2', _s[550], 5e-4, tol=1e-9)

<details>
<summary><b>💡 Hint</b></summary>

For $t<$ warmup: `lr_max * t / warmup`. Otherwise with $p = (t - \text{warmup})/(\text{total} - \text{warmup})$: `lr_min + (lr_max - lr_min) * 0.5 * (1 + cos(pi p))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is the de-facto default schedule for [[Transformers]] and many vision models. Variants: linear decay instead of cosine, a non-zero floor `lr_min`,
or "warm-up + stable + decay" (WSD) for open-ended training.
```python
import matplotlib.pyplot as plt; plt.plot(sched); plt.xlabel('step'); plt.ylabel('lr')
```

```python
def lr_at(t, lr_max, warmup, total, lr_min=0.0):
    if t < warmup:
        return lr_max * t / warmup
    p = (t - warmup) / (total - warmup)
    return lr_min + (lr_max - lr_min) * 0.5 * (1 + np.cos(np.pi * p))

sched = [lr_at(t, 1e-3, 100, 1000) for t in range(1001)]
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Optimizers](Optimizers.md).*